In [1]:
%pip install koreanize-matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import koreanize_matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, root_mean_squared_error

# pandas, seaborn 라이브러리 사용 옵션 설정
pd.set_option("display.max_columns", 100) #  최대 100개 열까지 표시한다.
pd.set_option("display.max_rows", 100) # 최대 100개 행까지 표시한다.
pd.set_option("display.width", 140) # 표의 출력 너비를 140자로 설정해 줄바꿈을 줄인다.
# seaborn 그래프의 기본 모양을 흰 배경과 격자선으로 설정하고, 노트북에 맞는 글자·요소 크기를 사용한다.
sns.set_theme(style="whitegrid", context="notebook")

In [3]:
df = pd.read_csv('train.csv')
X = df.drop(columns=['Id', 'SalePrice'])
y = df['SalePrice']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=1)

print(X_train.shape)
print(X_val.shape)
print(y_train.shape)
print(y_val.shape)

(1168, 79)
(292, 79)
(1168,)
(292,)


In [4]:
# 결측치 처리 1) "None", "Unknown"
# 전체 결측치 수는
print('전 X_train 결측치:',X_train.isna().sum().sum())
print('전 X_val 결측치:',X_val.isna().sum().sum())

전 X_train 결측치: 6247
전 X_val 결측치: 1582


In [5]:
# 1) 결측치를 "None"값으로 대처하는 컬럼들
na_cols =[
    'Alley','BsmtQual','BsmtCond','BsmtFinType1','FireplaceQu','GarageType','GarageFinish','GarageQual','GarageCond','PoolQC','Fence','MiscFeature'
]

for nc in na_cols: # 12번 반복
  X_train[nc] = X_train[nc].fillna('None')
  X_val[nc] = X_val[nc].fillna('None')

In [6]:
print('전 X_train 결측치:',X_train.isna().sum().sum())
print('전 X_val 결측치:',X_val.isna().sum().sum())

전 X_train 결측치: 1027
전 X_val 결측치: 270


In [7]:
# 2) BsmtExposure, BsmtFinType2는 BsmtExposure결측치이고, TotalBsmtSF값이 0이면 "None", 아니면 "Unknown"

# BsmtExposure 결측치 -> 'None' / 'Unknown'
cond = X_train['BsmtExposure'].isna() & (X_train['TotalBsmtSF']==0)
X_train.loc[cond, 'BsmtExposure'] = "None"
X_train.loc[X_train['BsmtExposure'].isna(), 'BsmtExposure'] = "Unknown"

cond = X_val['BsmtExposure'].isna() & (X_val['TotalBsmtSF']==0)
X_val.loc[cond, 'BsmtExposure'] = "None"
X_val.loc[X_val['BsmtExposure'].isna(), 'BsmtExposure'] = "Unknown"



# BsmtFinType2 결측치 -> 'None' / 'Unknown'
cond = X_train['BsmtFinType2'].isna() & (X_train['TotalBsmtSF']==0)
X_train.loc[cond, 'BsmtFinType2'] = "None"
X_train.loc[X_train['BsmtFinType2'].isna(), 'BsmtFinType2'] = "Unknown"

cond = X_val['BsmtFinType2'].isna() & (X_val['TotalBsmtSF']==0)
X_val.loc[cond, 'BsmtFinType2'] = "None"

X_val.loc[X_val['BsmtFinType2'].isna(), 'BsmtFinType2'] = "Unknown"


# 3) MasVnrType 결측치도 동일 방식으로 처리
cond = X_train['MasVnrType'].isna() & (X_train['MasVnrArea']==0)
X_train.loc[cond, 'MasVnrType'] = 'None'
X_train['MasVnrType'] = X_train['MasVnrType'].fillna('Unknown')

cond = X_val['MasVnrType'].isna() & (X_val['MasVnrArea']==0)
X_val.loc[cond, 'MasVnrType'] = 'None'
X_val['MasVnrType'] = X_val['MasVnrType'].fillna('Unknown')

In [8]:
print('전 X_train 결측치:',X_train.isna().sum().sum())
print('전 X_val 결측치:',X_val.isna().sum().sum())

전 X_train 결측치: 279
전 X_val 결측치: 70


In [9]:
# 4) GarageYrBlt은 GarageType이 None인 행에서 결측이면 0으로 처리한다. 이는 차고가 없는 집에 차고 건축연도가 없다는 상태를 표현한다.

# display(X_train.loc[X_train['GarageYrBlt'].isna(), ['GarageYrBlt','GarageType']])


cond = X_train['GarageYrBlt'].isna() & (X_train['GarageType'] == 'None')
X_train.loc[cond, 'GarageYrBlt'] = 0

cond = X_val['GarageYrBlt'].isna() & (X_val['GarageType'] == 'None')
X_val.loc[cond, 'GarageYrBlt'] = 0

print('후 X_train 결측치:',X_train.isna().sum().sum())
print('후 X_val 결측치:',X_val.isna().sum().sum())


후 X_train 결측치: 218
후 X_val 결측치: 50


In [10]:
na_train = X_train.isna().sum()
display(na_train[na_train > 0])

na_val = X_val.isna().sum()
display(na_val[na_val > 0])

LotFrontage    209
MasVnrArea       8
Electrical       1
dtype: int64

LotFrontage    50
dtype: int64

In [11]:
# 남은 결측치
# 1) LotFrontage	- 중앙값 - 클래스별 중앙값
#    남은 na :  MasVnrArea(8), Electrical(1)

# 2) MasVnrArea
# 3) Electrical

In [12]:
X_train.loc[ X_train['LotFrontage'].isna(), ['LotFrontage', 'Neighborhood']]

,LotFrontage,Neighborhood
131,NaN,Gilbert
1429,NaN,NWAmes
720,NaN,StoneBr
404,NaN,Gilbert
1003,NaN,NWAmes
...,...,...
1417,NaN,NoRidge
1110,NaN,Gilbert
580,NaN,NAmes
668,NaN,Sawyer


In [13]:
X_train['Neighborhood'].unique()

<StringArray>
['Edwards', 'OldTown', 'CollgCr', 'SawyerW', 'Gilbert',   'NAmes', 'Mitchel',  'NWAmes', 'Somerst',  'IDOTRR', 'NPkVill', 'NridgHt',
 'Crawfor', 'StoneBr', 'Blmngtn', 'NoRidge', 'BrkSide',  'Sawyer',  'BrDale', 'MeadowV',  'Timber', 'ClearCr',   'SWISU', 'Veenker',
 'Blueste']
Length: 25, dtype: str

In [14]:
by_neighborhood =X_train.groupby('Neighborhood')['LotFrontage'].median()
print(len(by_neighborhood))
# display(by_neighborhood)

print(len(X_train['LotFrontage']))

25
1168


In [15]:
# 타겟열.fillna(시리즈) : 타겟열의 길이와 시리즈길이가 같아야 결측치 자리에 같은 행의 값이 대체
# 그룹핑되어 25개의 길이를 X_train['Neighborhood']길이에 맵핑
mapping_values = X_train['Neighborhood'].map(by_neighborhood)
mapping_values2 = X_val['Neighborhood'].map(by_neighborhood)
print(len(mapping_values))
#display(mapping_values)

1168


In [16]:
# 맵핑후 길이를 같게한 시리즈를 fillna 매개값으로 사용
X_train['LotFrontage'] = X_train['LotFrontage'].fillna(mapping_values)
X_val['LotFrontage'] = X_val['LotFrontage'].fillna(mapping_values2)


In [17]:
na_train = X_train.isna().sum()
display(na_train[na_train > 0])

na_val = X_val.isna().sum()
display(na_val[na_val > 0])

MasVnrArea    8
Electrical    1
dtype: int64

Series([], dtype: int64)

In [18]:
X_train['MasVnrArea'] = X_train['MasVnrArea'].fillna(0)
X_val['MasVnrArea'] = X_val['MasVnrArea'].fillna(0)

In [19]:
X_train['Electrical'].unique() # 범주형이고 결측치 하나이니 최빈값으로 대체하자

<StringArray>
['SBrkr', 'FuseA', 'FuseF', nan, 'FuseP', 'Mix']
Length: 6, dtype: str

In [20]:
na_train = X_train.isna().sum()
display(na_train[na_train > 0])

na_val = X_val.isna().sum()
display(na_val[na_val > 0])

Electrical    1
dtype: int64

Series([], dtype: int64)

In [21]:
na_value = X_train['Electrical'].mode()[0] # 반환값은 순위가 같을 수 있기에 스칼라가 아닌 시리즈를 반환, 첫번째 값만 얻고자 한다면 []사용
X_train['Electrical'] = X_train['Electrical'].fillna(na_value)
X_val['Electrical'] = X_val['Electrical'].fillna(na_value)

In [22]:
print(X_train.isna().sum().sum()) # --> 0
print(X_val.isna().sum().sum()) # --> 0

0
0


In [23]:
# 결측치 대체

In [24]:
# RandomForestRegressor -> 범주 인코딩 필요, 수치 스케일링 필요
# 수치인데 범주에 가까운 것들 : MSSubClass, OverallQual, OverallCond, MoSold
# 수치들도 인코딩 필요하더면 -> 데이터 타입을 object, str, category 등으로 변경 후 인코딩 진행
# 0, 1, 2 있지만 순서(서열)형이면 OneHotEncoding보다 그대로 두는게 좋다

cols = ['MSSubClass', 'OverallQual', 'OverallCond', 'MoSold']

for c in cols:
  print(X_train[c].unique())


[ 90 190  20  60  80  50 160  30 120  70  40  85 180  45  75]
[ 5  4  7  6  8 10  9  2  3  1]
[7 5 6 4 8 3 9 2]
[ 9  8  7  1  4  6  5  3 12  2 11 10]


In [25]:
# 데이터셋 탐색결과에서 MSSubClass 서열(순서)없는 값이다 -> OneHotEncoding 적용 -> object/str 타입으로 변경하자
X_train['MSSubClass'] = X_train['MSSubClass'].astype(str)
X_val['MSSubClass'] = X_val['MSSubClass'].astype(str)

In [26]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 1168 entries, 921 to 1061
Data columns (total 79 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1168 non-null   str    
 1   MSZoning       1168 non-null   str    
 2   LotFrontage    1168 non-null   float64
 3   LotArea        1168 non-null   int64  
 4   Street         1168 non-null   str    
 5   Alley          1168 non-null   str    
 6   LotShape       1168 non-null   str    
 7   LandContour    1168 non-null   str    
 8   Utilities      1168 non-null   str    
 9   LotConfig      1168 non-null   str    
 10  LandSlope      1168 non-null   str    
 11  Neighborhood   1168 non-null   str    
 12  Condition1     1168 non-null   str    
 13  Condition2     1168 non-null   str    
 14  BldgType       1168 non-null   str    
 15  HouseStyle     1168 non-null   str    
 16  OverallQual    1168 non-null   int64  
 17  OverallCond    1168 non-null   int64  
 18  YearBuilt      1168 no

In [27]:
# OneHotEncoding
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore').set_output(transform='pandas')

In [28]:
obj_cols = X_train.select_dtypes('object').columns
# num_cols = X_train.select_dtypes('number').columns
obj_cols

C:\Users\user\AppData\Local\Temp\ipykernel_16852\4121200092.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = X_train.select_dtypes('object').columns


Index(['MSSubClass', 'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood',
       'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition'],
      dtype='str')

In [29]:
X_train_en = encoder.fit_transform(X_train[obj_cols])
X_val_en = encoder.transform(X_val[obj_cols])

In [30]:
# X_train[num_cols] == X_train.drop(columns=obj_cols)
X_train_pre = pd.concat([X_train.drop(columns=obj_cols), X_train_en], axis=1)
X_val_pre = pd.concat([X_val.drop(columns=obj_cols), X_val_en], axis=1)

In [31]:
print('원핫인코딩 전 X_train.shape:', X_train.shape)
print('원핫인코딩 후 X_train_pre.shape:', X_train_pre.shape)

print('원핫인코딩 전 X_val.shape:', X_val.shape)
print('원핫인코딩 후 X_val_pre.shape:', X_val_pre.shape)

원핫인코딩 전 X_train.shape: (1168, 79)
원핫인코딩 후 X_train_pre.shape: (1168, 316)
원핫인코딩 전 X_val.shape: (292, 79)
원핫인코딩 후 X_val_pre.shape: (292, 316)


In [32]:
# 학습 -> 평가 -> 예측(test.csv)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, root_mean_squared_error

In [33]:
model = RandomForestRegressor(
    n_estimators=500, # 트리계수
    min_samples_leaf=1,
    random_state=1,
    n_jobs=-1
)

In [34]:
# X_train_pre , y_train 학습
# X_val_pre -> 예측치 -> y_val 비교해서 r2, rmse 평가지표 확인

model.fit(X_train_pre, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"ma

In [35]:
y_pred = model.predict(X_val_pre) # y_val

In [36]:
r2 = r2_score(y_val, y_pred)
rmse = root_mean_squared_error(y_val, y_pred)

In [37]:
print(r2)
print(rmse)

0.8882839192262354
28226.80928347678


0.8799339194540885

29262.6808049898

In [38]:
# test.csv 예측값 출력
# 모델 저장
# 인코더 저장....

In [39]:
# 비교 학습을 한다.
# y에 로그
# 특성공학(3개의 변수 추가)
# 여러 모델을 혼합해서 새로운 모델

# 하이퍼 파라미터 비교
# 다른 모델과도 비교
# 교차검증(k-fold,....)